# Data Inventory\n\n**Purpose:** List data files with schema info\n\n**Features:** Fast schema reads, check required columns

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import yaml, os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'lib'))
from utils import setup_notebook_environment, get_machine_id, load_config

project_root = setup_notebook_environment()
os.chdir(project_root)
print("="*60)
print("  DATA INVENTORY")
print("="*60)
schemas = {}

## 1. Machine Configuration

## 2. Schema Analysis

In [ ]:
machine_id = get_machine_id()
cfg = load_config(str(Path(config_path) / "config.yaml"))
paths = cfg['machines'][f'PC{machine_id}']['paths']

files = {
    'Master': Path(paths['master_data_path']) / cfg['data']['master_file'],
    'Aux': Path(paths['aux_data_path']) / cfg['data']['aux_file'],
    'GLM': Path(paths['aux_data_path']) / cfg['data']['control_model_file']
}

print(f"\\nMachine: PC{machine_id}\\n")
for name, path in files.items():
    if not path.exists():
        print(f"{name:10s}: No")
        continue
    pf = pq.ParquetFile(path)
    num_rows = pf.metadata.num_rows
    num_cols = len(pf.schema_arrow.names)
    schemas[name] = {'columns': pf.schema_arrow.names, 'num_rows': num_rows}
    size_mb = path.stat().st_size / (1024*1024)
    print(f"{name:10s}: Yes  {num_rows:>10,} rows x {num_cols:>4} cols  {size_mb:>8.1f} MB")

## 3. Required Columns

In [ ]:
coverage_cols = {
    'coll': {'Master': ['ee_coll_imps'], 'Aux': ['pp_coll'], 'GLM': ['pred_pp_coll']},
    'comp': {'Master': ['ee_comp_imps'], 'Aux': ['pp_comp'], 'GLM': ['pred_pp_comp']},
    'pip': {'Master': ['ee_pip_imps'], 'Aux': ['pp_pip'], 'GLM': ['pred_pp_pip']},
    'liab': {'Master': ['ee_bi_imps', 'ee_pd_imps'], 'Aux': ['pp_bi', 'pp_pd'], 'GLM': ['pred_pp_liab_1p', 'pred_pp_liab_3p']}
}

print("\\nRequired Columns:\\n" + "-"*60)
for cov, req in coverage_cols.items():
    print(f"\\n{cov.upper()}:")
    for file_name, cols in req.items():
        if schemas.get(file_name):
            for col in cols:
                status = "Yes" if col in schemas[file_name]['columns'] else "No"
                print(f"  {file_name:10s} {col:25s}: {status}")

## 4. GLM Predictions

In [ ]:
if schemas.get('GLM'):
    pred_cols = [c for c in schemas['GLM']['columns'] if c.startswith('pred_')]
    print(f"\\nGLM Predictions ({len(pred_cols)}):")
    for col in sorted(pred_cols):
        print(f"  - {col}")
    expected = ['pred_pp_coll', 'pred_pp_comp', 'pred_pp_pip', 'pred_pp_liab_1p', 'pred_pp_liab_3p']
    missing = [c for c in expected if c not in pred_cols]
    if missing:
        print(f"\\nMissing:")  
        for col in missing:
            print(f"  - {col}")
else:
    print("GLM file not found")